In [1]:
#!pip install -qq scikit-learn==1.6.1

In [2]:
!pip install category_encoders

In [3]:
from category_encoders import TargetEncoder

In [4]:
from tqdm import tqdm
from itertools import combinations

import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold

import lightgbm as lgb

import warnings
warnings.simplefilter('ignore')

In [5]:
def feature_eng(df):
    podc_dict = {'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47}
    genr_dict = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
    week_dict = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
    time_dict = {'Morning': 0, 'Afternoon': 1, 'Evening': 2, 'Night': 3}
    sent_dict = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
    
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype('category')
    
    df['Genre'] = df['Genre'].replace(genr_dict)
    df['Podcast_Name'] = df['Podcast_Name'].replace(podc_dict)
    df['Publication_Day'] = df['Publication_Day'].replace(week_dict)
    df['Publication_Time'] = df['Publication_Time'].replace(time_dict)
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(sent_dict)
    
    df['Genre'] = df['Genre'].astype('category')
    df['Podcast_Name'] = df['Podcast_Name'].astype('category')
    df['Publication_Day'] = df['Publication_Day'].astype('category')
    df['Publication_Time'] = df['Publication_Time'].astype('category')
    df['Episode_Sentiment'] = df['Episode_Sentiment'].astype('category')
    
    df = df.drop(columns=['Episode_Title'])
    return df

In [6]:
df_train = pd.read_csv('/kaggle/input/playground-series-s5e4/train.csv', index_col='id')
df_train = feature_eng(df_train)

df_test = pd.read_csv('/kaggle/input/playground-series-s5e4/test.csv', index_col='id')
df_test = feature_eng(df_test)

df_subm = pd.read_csv('/kaggle/input/playground-series-s5e4/sample_submission.csv', index_col='id')

In [7]:
encode_columns = ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment', 'Publication_Day', 'Publication_Time']
pair_size = [2, 3, 4]

for r in pair_size:
    for cols in tqdm(list(combinations(encode_columns, r))):
        new_col_name = '_'.join(cols)
        
        df_train[new_col_name] = df_train[list(cols)].astype(str).agg('_'.join, axis=1)
        df_train[new_col_name] = df_train[new_col_name].astype('category')
        
        df_test[new_col_name] = df_test[list(cols)].astype(str).agg('_'.join, axis=1)
        df_test[new_col_name] = df_test[new_col_name].astype('category')

100%|██████████| 35/35 [04:44<00:00,  8.13s/it]


In [8]:
X = df_train.drop(columns=['Listening_Time_minutes'])
y = df_train['Listening_Time_minutes']

In [9]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

val_scores = []
models = []
oof_preds = np.zeros(X.shape[0])

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Training Fold {fold + 1}")

    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    lgb_train = lgb.Dataset(X_train_fold, y_train_fold)
    lgb_val = lgb.Dataset(X_val_fold, y_val_fold)

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'feature_fraction': 0.85,
        'bagging_fraction': 0.85,
        'seed': 42,
        'verbose': -1
    }

    model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)  # This replaces verbose_eval
        ]
    )

    val_preds = model.predict(X_val_fold)
    fold_rmse = mean_squared_error(y_val_fold, val_preds, squared=False)
    val_scores.append(fold_rmse)
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.4f}")

    oof_preds[val_idx] = val_preds
    models.append(model)

# Overall validation score
final_rmse = mean_squared_error(y, oof_preds, squared=False)
print(f"\nOverall CV RMSE: {final_rmse:.4f}")

Training Fold 1
Training until validation scores don't improve for 50 rounds
[100]	training's rmse: 12.1964	valid_1's rmse: 12.9516
[200]	training's rmse: 11.5943	valid_1's rmse: 12.9336
Early stopping, best iteration is:
[176]	training's rmse: 11.7054	valid_1's rmse: 12.9316
Fold 1 RMSE: 12.9316
Training Fold 2
Training until validation scores don't improve for 50 rounds
[100]	training's rmse: 12.1977	valid_1's rmse: 13.0089
[200]	training's rmse: 11.5964	valid_1's rmse: 12.9746
Early stopping, best iteration is:
[220]	training's rmse: 11.5136	valid_1's rmse: 12.9731
Fold 2 RMSE: 12.9731
Training Fold 3
Training until validation scores don't improve for 50 rounds
[100]	training's rmse: 12.1866	valid_1's rmse: 13.0169
[200]	training's rmse: 11.5923	valid_1's rmse: 12.9983
Early stopping, best iteration is:
[183]	training's rmse: 11.6665	valid_1's rmse: 12.9957
Fold 3 RMSE: 12.9957
Training Fold 4
Training until validation scores don't improve for 50 rounds
[100]	training's rmse: 12.187

In [10]:
X_test = df_test.copy()

test_preds = np.mean(
    [model.predict(X_test, num_iteration=model.best_iteration) for model in models],
    axis=0
)

sample_submission = pd.read_csv("/kaggle/input/playground-series-s5e4/sample_submission.csv")

sample_submission["Listening_Time_minutes"] = test_preds

sample_submission.to_csv("submission.csv", index=False)

print("✅ Submission file overwritten successfully.")

✅ Submission file overwritten successfully.
